<a href="https://colab.research.google.com/github/vxn09/VivianNguyen_1002410401_FinalProject/blob/main/VivianNguyen__1002410401_FinalProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn, scipy
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram
from IPython.display import display
from pathlib import Path
import json

from google.colab import drive
file_path = "/content/drive/MyDrive/GSE295523_gene_count_matrix.txt.gz"

pylori_raw = pd.read_csv(file_path, sep="\t", header=0, index_col=0, compression="gzip", encoding="latin1")




In [ ]:
file_path = "/content/drive/MyDrive/GSE295523_gene_count_matrix.txt.gz"

raw_data = pd.read_csv(
    file_path,
    sep="\t",
    index_col=0,
    compression="gzip"
)

print("Count matrix dimensions:", raw_data.shape)

display(raw_data.head())

print(raw_data.columns.tolist())



Count matrix dimensions: (63086, 24)


,NC4,NC5,NC6,AOE4,AOE5,AOE6,NC10,NC11,NC12,GOE10,...,NC3,HOE1,HOE2,HOE3,NC7,NC8,NC9,MOE7,MOE8,MOE9
gene_id,,,,,,,,,,,,,,,,,,,,,
ENSG00000282222,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ENSG00000282221,81,120,185,130,166,125,0,0,0,0,...,0,0,0,0,193,227,168,589,493,548
ENSG00000212040,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ENSG00000110514,3687,3963,4723,6199,4954,5564,5130,5900,5723,5619,...,4580,3622,3653,3908,4420,4392,5020,4165,4735,4750
ENSG00000287159,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


['NC4', 'NC5', 'NC6', 'AOE4', 'AOE5', 'AOE6', 'NC10', 'NC11', 'NC12', 'GOE10', 'GOE11', 'GOE12', 'NC1', 'NC2', 'NC3', 'HOE1', 'HOE2', 'HOE3', 'NC7', 'NC8', 'NC9', 'MOE7', 'MOE8', 'MOE9']


In [ ]:
ges_samples = [
    "NC10",
    "NC11",
    "NC12",
    "GOE10",
    "GOE11",
    "GOE12"
]
ges_counts = raw_data[ges_samples].copy()

print("GES-1 count matrix dimensions:", ges_counts.shape)
display(ges_counts.head())

GES-1 count matrix dimensions: (63086, 6)


,NC10,NC11,NC12,GOE10,GOE11,GOE12
gene_id,,,,,,
ENSG00000282222,0,0,0,0,1,0
ENSG00000282221,0,0,0,0,0,0
ENSG00000212040,0,0,0,0,0,0
ENSG00000110514,5130,5900,5723,5619,4951,5721
ENSG00000287159,0,0,0,0,0,0


In [ ]:
counts = ges_counts.T

print(counts.shape)
display(counts.head(6))

(6, 63086)


gene_id,ENSG00000282222,ENSG00000282221,ENSG00000212040,ENSG00000110514,ENSG00000287159,ENSG00000086015,ENSG00000211769,ENSG00000211768,ENSG00000211767,ENSG00000211766,...,ENSG00000280310,ENSG00000280311,ENSG00000280312,ENSG00000182872,ENSG00000182873,ENSG00000182870,ENSG00000143458,ENSG00000256751,ENSG00000200935,ENSG00000287855
NC10,0,0,0,5130,0,5296,0,0,0,0,...,3,3,0,7368,114,19,1547,12,0,19
NC11,0,0,0,5900,0,5649,0,0,0,0,...,6,14,0,7609,149,9,1400,41,0,18
NC12,0,0,0,5723,0,5824,0,0,0,0,...,14,9,0,6887,93,2,1278,4,0,20
GOE10,0,0,0,5619,0,9876,0,0,0,0,...,2,17,0,11805,173,84,2120,57,0,37
GOE11,1,0,0,4951,0,10237,0,0,0,0,...,10,4,0,12145,169,35,2208,49,0,60
GOE12,0,0,0,5721,0,11023,0,0,0,0,...,22,1,0,13408,159,19,2558,37,0,58


In [ ]:
metadata = pd.DataFrame({
    "condition": [
        "control",
        "control",
        "control",
        "infected",
        "infected",
        "infected"
    ]
}, index=ges_samples)

display(metadata)

print("Sample names match:", all(counts.index == metadata.index))

print("Missing values:", counts.isna().sum().sum())
print("Negative values:", (counts < 0).sum().sum())
print("Are all values integers:", (counts % 1 == 0).all().all())

display(counts.iloc[:5, :5])

,condition
NC10,control
NC11,control
NC12,control
GOE10,infected
GOE11,infected
GOE12,infected


Sample names match: True
Missing values: 0
Negative values: 0
Are all values integers: True


gene_id,ENSG00000282222,ENSG00000282221,ENSG00000212040,ENSG00000110514,ENSG00000287159
NC10,0,0,0,5130,0
NC11,0,0,0,5900,0
NC12,0,0,0,5723,0
GOE10,0,0,0,5619,0
GOE11,1,0,0,4951,0


In [ ]:
!pip -q install rpy2
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R -i ges_counts

# --- Package installation (only installs if missing) ---
if (!requireNamespace("BiocManager", quietly=TRUE)) {
  install.packages("BiocManager", repos="https://cloud.r-project.org")
}
if (!requireNamespace("DESeq2", quietly=TRUE)) {
  BiocManager::install("DESeq2", ask=FALSE)
}

# --- Load required libraries into the R environment ---
library(DESeq2)

# --- Define the sample names and metadata ---
samples <- c("NC10", "NC11", "NC12", "GOE10", "GOE11", "GOE12")
condition <- factor(c("uninfected", "uninfected", "uninfected", "infected", "infected", "infected"))

coldata <- data.frame(row.names=samples, condition=condition)

# --- Convert input counts matrix ---
# Convert the imported Python pandas DataFrame to a matrix of integers
count_matrix <- as.matrix(ges_counts)
class(count_matrix) <- "integer"

# --- Create the DESeq2 dataset ---
dds <- DESeqDataSetFromMatrix(countData=count_matrix, colData=coldata, design=~condition)

# --- Set reference level and run DESeq2 analysis ---
# Relevel so "uninfected" is treated as the reference/baseline
dds$condition <- relevel(dds$condition, ref="uninfected")

# Perform DESeq2 analysis
dds <- DESeq(dds)

# --- Extract and organize results ---
# This contrasts infected vs uninfected
res <- results(dds, contrast=c("condition", "infected", "uninfected"))

# Order results by adjusted p-value (FDR)
res <- res[order(res$padj), ]

# --- Save and summarize ---
write.csv(as.data.frame(res), file="DESeq2_results.csv")
summary(res)


out of 33427 with nonzero total read count
adjusted p-value < 0.1
LFC > 0 (up)       : 6261, 19%
LFC < 0 (down)     : 5866, 18%
outliers [1]       : 34, 0.1%
low counts [2]     : 7414, 22%
(mean count < 2)
[1] see 'cooksCutoff' argument of ?results
[2] see 'independentFiltering' argument of ?results



Loading required package: S4Vectors
Loading required package: stats4
Loading required package: BiocGenerics
Loading required package: generics

Attaching package: ‘generics’

The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union


Attaching package: ‘BiocGenerics’

The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs

The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Attaching package: ‘S4Vectors’

The following object is masked from ‘package:utils’:

    findMatches

The following object